In [ ]:
import torch
torch.cuda.is_available()
print("this should update")

In [ ]:
# Im just adding stuff so this starts synchronizing
hello= "hello"
print(hello)

In [ ]:
import semsimlib
from utils import DATA_DIR
m = semsimlib.DependencyMatrix(DATA_DIR / "verb_relation_counts_select.txt")
m.calculatePMI()
m_np = semsimlib.NPMatrix(m)
m_nmf = semsimlib.NMFMatrix(m_np, rdim=100)
m_nmf.compute(niters=50)
m_nmf.normalize()


In [ ]:
for i in range(m_nmf.rdim):
    print(i)
    print(m_nmf.getTopWordsDim(i, 10))
    print("\n")

In [ ]:
for instance in ["onderscheiden", "winnen", "vliegen", "lachen", "sterven", "willen", "huwen"]:
    if instance in m_nmf.features or instance in m_nmf.instances:
        print(f"most similar terms to {instance}:")
        print(m_nmf.calculateMostSimilar(instance, 5))
    
    else:
        print(f"model has not seen {instance}")

In [ ]:
m_nmf.calculateMostSimilar()

In [ ]:
feat_dict = {}
for feature in m_nmf.features:
    feat = feature.split("#")[1]
    if feat in feat_dict.keys():
        feat_dict[feat].append(feature)
    else:
        feat_dict[feat] = [feature]
m_nmf.calculateMostSimilar("obl#1922", 5)

In [ ]:
import pickle

# Load the tuple of matrices that were calculated here below (takes a long time)
with open(DATA_DIR/'matrices.pkl', 'rb') as f:
    m, m_nmf = pickle.load(f)

# test if this worked:
print("top words dim 5:", m_nmf.getTopWordsDim(5), "\n")
print("similar words to mens:", m_nmf.calculateMostSimilar("mens", 5))

In [ ]:
import datasets
from datasets import load_dataset

ds = load_dataset("BramVanroy/wikipedia_culturax_dutch", "100M", split="train")

In [ ]:
# we save the text column to a file corpus.txt
with open(DATA_DIR/"corpus.txt", "w", encoding="utf-8") as f:
    for text in ds["text"]:
        f.write(text.replace("\n", " ") + "\n")

In [ ]:
# we print the head of corpus.txt
with open(DATA_DIR/"corpus.txt", "r", encoding="utf-8") as f:
    for _ in range(5):
        print(f.readline().strip())

In [ ]:
import semsimlib
m = semsimlib.WindowMatrix(DATA_DIR'clean_corpus.txt')  # cleaned corpus of some bot-generated wiki entries (1/3)

In [ ]:
m.calculatePMI()

In [ ]:
m_np = semsimlib.NPMatrix(m)

In [ ]:
m_nmf = semsimlib.NMFMatrix(m_np,rdim=50)

In [ ]:
m_nmf.compute()

In [ ]:
m_nmf.normalize()

In [ ]:
for i in range(50):
    print(i)
    print(m_nmf.getTopWordsDim(i))
    print("\n")

In [ ]:
import numpy
for instance in ["zee", "Tom", "werk", "hout", "man", "vrouw"]:
    if instance in m_nmf.features:
        print(f"most similar terms to {instance}:")
        print(m_nmf.calculateMostSimilar(instance, 5))
    else:
        print(f"model has not seen {instance}")

In [ ]:
import pickle

# we save the m and m_nmf to a pickle to load them in later
matrices = (m, m_nmf)
with open(DATA_DIR/"matrices.pkl", "wb") as p:
    pickle.dump(matrices, p)

In [ ]:
import numpy
for instance in ["zee", "Tom", "werk", "hout", "man", "vrouw"]:
    if instance in m_nmf.features:
        print(f"most similar terms to {instance}:")
        print(m_nmf.calculateMostSimilar(instance, 5))
    else:
        print(f"model has not seen {instance}")

In [ ]:
import datasets
from datasets import load_dataset

ds = load_dataset("BramVanroy/wikipedia_culturax_dutch", "10k", split="train")

In [ ]:
with open(DATA_DIR/"clean_corpus.txt", "r", encoding="utf-8") as f:
    corpus = f.read()
corpus = corpus[:10000]

In [ ]:
import spacy
nlp = spacy.load("nl_core_news_sm")
doc = nlp(corpus)

In [ ]:
i = 0
while i < 5:
    for sent in doc.sents:
        for token in sent:
            print(token.text, token.pos_, token.dep_)
    i += 1

# Tensor experiments
We use the created tensors and vocabularies as created in tensor population.ipynb

In [ ]:
import torch
import pickle
# we load the tensor and vocabularies
# we save the respective vocabularies as well
# vocab = {
#     "vocab_v": vocab_v,
#     "vocab_s": vocab_s,
#     "vocab_o": vocab_o,
#     "v2i": v2i,
#     "s2i": s2i,
#     "o2i": o2i
# }
with open(DATA_DIR/"tensors/vocabularies.pkl", "rb") as f:
    vocab = pickle.load(f)
vocab_v = vocab["vocab_v"]
vocab_s = vocab["vocab_s"]
vocab_o = vocab["vocab_o"]
v2i = vocab["v2i"]
s2i = vocab["s2i"]
o2i = vocab["o2i"]

# we load in the different tensors
with open(DATA_DIR/"tensors/tensor_counting.pkl", "rb") as f:
    tensor_counting = pickle.load(f)
with open(DATA_DIR/"tensors/tensor_sc.pkl", "rb") as f:
    tensor_sc = pickle.load(f)
with open(DATA_DIR/"tensors/tensor_sii.pkl", "rb") as f:
    tensor_sii = pickle.load(f)

In [ ]:
# we perform Tucker Decomposition using tensorly
import tensorly as tl
from tensorly.decomposition import tucker, non_negative_tucker
import time
tl.set_backend('pytorch')
tic = time.time()
core, factors = tucker(tl.tensor(tensor_sc), rank=100, n_iter_max=1000, init='random')
tucker_reconstruction = tl.tucker_to_tensor((core, factors))
time_mu = time.time() - tic
print("Time taken for Tucker decomposition:", time_mu)

In [ ]:
tic = time.time()
core_sc, factors_sc = non_negative_tucker(
    tensor_sc, rank=100, tol=1e-12, n_iter_max=100, init='random'
)
tucker_reconstruction_mu = tl.tucker_to_tensor((core_sc, factors_sc))
time_mu = time.time() - tic
print("Time taken for Non-negative Tucker decomposition:", time_mu)


In [ ]:
tic = time.time()
core_sii, factors_sii = non_negative_tucker(
    tensor_sii, rank=100, tol=1e-12, n_iter_max=100, init='random'
)
tucker_reconstruction_sii = tl.tucker_to_tensor((core_sii, factors_sii))
time_mu = time.time() - tic
print("Time taken for Non-negative Tucker decomposition (SII):", time_mu)

In [ ]:
# we save to a pickle
with open(DATA_DIR/"tensors/sc_tucker_decomposition.pkl", "wb") as f:
    pickle.dump((core_sc, factors_sc), f)
with open(DATA_DIR/"tensors/sii_tucker_decomposition.pkl", "wb") as f:
    pickle.dump((core_sii, factors_sii), f)

In [ ]:
# we check average error:
error = tl.norm(tensor_sc - tucker_reconstruction) / tl.norm(tensor_sc)
error_non_negative_sc = tl.norm(tensor_sc - tucker_reconstruction_mu) / tl.norm(tensor_sc)
error_non_negative_sii = tl.norm(tensor_sii - tucker_reconstruction_sii) / tl.norm(tensor_sii)
print("Relative error (Tucker):", error.item())
print("Relative error (Non-negative Tucker SC):", error_non_negative_sc.item())
print("Relative error (Non-negative Tucker SII):", error_non_negative_sii.item())


In [ ]:
# we try to interpret the factors ~ latent dimensions
# we normalize the factors for easier interpretation
factor_verb = factors_sii[0]
factor_subj = factors_sii[1]
factor_obj = factors_sii[2]

In [ ]:
for i in range(factor_verb.shape[1]):
    print(f"Dimension {i}:")
    # verbs
    top_values, top_indices = torch.topk(factor_verb[:, i], 10)
    top_verbs = [f"{vocab_v[idx]} ({top_values[j].item():.3f})" for j, idx in enumerate(top_indices)]
    # subjects
    top_values, top_indices = torch.topk(factor_subj[:, i], 10)
    top_subj = [f"{vocab_s[idx]} ({top_values[j].item():.3f})" for j, idx in enumerate(top_indices)]
    # objects
    top_values, top_indices = torch.topk(factor_obj[:, i], 10)
    top_obj = [f"{vocab_o[idx]} ({top_values[j].item():.3f})" for j, idx in enumerate(top_indices)]
    
    print("verbs:", top_verbs)
    print("subjects:", top_subj)
    print("objects:", top_obj)
    print("\n")

In [ ]:
# based on this idea, we can build up a cosine similarity function
from sklearn.metrics.pairwise import cosine_similarity
def verb_similarity(verb1, verb2):
    if verb1 not in v2i or verb2 not in v2i:
        print("One or both verbs not in vocabulary.")
        return None
    vec1 = factor_verb[v2i[verb1], :].unsqueeze(0).numpy()
    vec2 = factor_verb[v2i[verb2], :].unsqueeze(0).numpy()
    similarity = cosine_similarity(vec1, vec2)[0][0]
    return similarity

# we test it
print("Similarity between 'spelen' and 'winnen':", verb_similarity("spelen", "winnen"))
print("Similarity between 'spelen' and 'oproepen':", verb_similarity("spelen", "oproepen"))
print("Similarity between 'spelen' and 'verliezen':", verb_similarity("spelen", "verliezen"))
print("Similarity between 'winnen' and 'verliezen':", verb_similarity("winnen", "verliezen"))

In [ ]:
# we create a "top similar verbs" function
def top_similar_verbs(target_verb, top_n=5):
    if target_verb not in v2i:
        print(f"Verb '{target_verb}' not in vocabulary.")
        return []
    target_vector = factor_verb[v2i[target_verb], :].unsqueeze(0).numpy()
    similarities = []
    for verb in vocab_v:
        if verb == target_verb:
            continue
        verb_vector = factor_verb[v2i[verb], :].unsqueeze(0).numpy()
        sim = cosine_similarity(target_vector, verb_vector)[0][0]
        similarities.append((verb, sim))
    similarities.sort(key=lambda x: x[1], reverse=True)
    return similarities[:top_n]


In [ ]:
# we test it
test = ["spelen", "winnen", "verliezen", "geven", "nemen", "steken", "eisen", "leven"]
for verb in test:
    print(f"Top similar verbs to '{verb}':", top_similar_verbs(verb, 5))

# Core tensor shift modelling
By obtaining the dimension (k) values of each mode, we can retrace the appropriate point in the core tensor
Then, we can calculate the "distance" between two sentences in this core tensor as a "shift" in meaning

## Effect of normalizing the data
There is an effect: In the weighted version, some quantile information gets changed (we already normalize in the function)

In [ ]:
import itertools
import numpy as np

core = core_sc.numpy()

def create_core_projection(test_tuple, weighted=False):
    verb_idx = v2i[test_tuple[0]]
    subj_idx = s2i[test_tuple[1]]
    obj_idx = o2i[test_tuple[2]]
    # we get their corresponding values in each dimension
    verb_dims = factor_verb[verb_idx, :].numpy()
    subj_dims = factor_subj[subj_idx, :].numpy()
    obj_dims = factor_obj[obj_idx, :].numpy()
    
    # we project this to a distribution in a core tensor like space
    core_projection = np.zeros(core.shape)
    for i, j, k in itertools.product(range(core.shape[0]), range(core.shape[1]), range(core.shape[2])):
        core_projection[i, j, k] = verb_dims[i] * subj_dims[j] * obj_dims[k]
    
    if weighted:
        core_projection = core_projection * core
    return core_projection
# Using some algebraic ideas, we can make the same function a lot faster
# --> Not actually populating the core tensor element by element, but using Einstein summation (implemented via np)
def core_projection_fast(test_tuple, weighted=True):
    verb_idx = v2i[test_tuple[0]]
    subj_idx = s2i[test_tuple[1]]
    obj_idx = o2i[test_tuple[2]]
    v = factor_verb[verb_idx, :].cpu().numpy()
    s = factor_subj[subj_idx, :].cpu().numpy()
    o = factor_obj[obj_idx, :].cpu().numpy()
    if weighted:
        # elementwise multiply the core by the outer-product weights
        # shape (R1,R2,R3): v[i]*s[j]*o[k] * core[i,j,k]
        proj = np.einsum('i,j,k,ijk->ijk', v, s, o, core)
    else:
        proj = np.einsum('i,j,k->ijk', v, s, o)
    return proj

def plot_tensor_plotly(tensor: np.ndarray,
                       title=None,
                       top_quantile=0.95,
                       n_isosurfaces=10,
                       show_cube=True,
                       show_grid=True,
                       cube_opacity=0.1,
                       show=True,
                       save_dir=None,
                       return_fig=False):
    """
    Render a 3D tensor as a transparent cube with only the highest values visible.

    - tensor: 3D numpy array (Z, Y, X) or (X, Y, Z) — any order is fine for scalar values
    - top_quantile: only values >= this quantile get an isosurface
    - n_isosurfaces: number of nested high-value surfaces to draw
    """
    assert tensor.ndim == 3, "tensor must be 3D"

    # Normalize to [0, 1] (keeps relative structure)
    vol = tensor.astype(np.float32)
    # vmin, vmax = float(np.nanmin(vol)), float(np.nanmax(vol))
    # if vmax == vmin:
    #     raise ValueError("Tensor has constant value; nothing to visualize.")
    # vol = (vol - vmin) / (vmax - vmin)

    # Choose isovals in the top tail
    q0 = np.quantile(vol, top_quantile)
    # Distribute a few levels between q0 and 1.0
    isovals = np.linspace(q0, 1.0, n_isosurfaces)

    Z, Y, X = vol.shape  # plotly expects x,y,z arrays but can index by grid
    fig = go.Figure()

    # Optional: translucent wireframe cube for context
    if show_cube:
        # Cube edges as line segments
        corners = np.array([
            [0, 0, 0], [X-1, 0, 0], [X-1, Y-1, 0], [0, Y-1, 0],
            [0, 0, Z-1], [X-1, 0, Z-1], [X-1, Y-1, Z-1], [0, Y-1, Z-1]
        ])
        edges = [
            (0,1),(1,2),(2,3),(3,0),
            (4,5),(5,6),(6,7),(7,4),
            (0,4),(1,5),(2,6),(3,7)
        ]
        xs, ys, zs = [], [], []
        for i,j in edges:
            xs += [corners[i,0], corners[j,0], None]
            ys += [corners[i,1], corners[j,1], None]
            zs += [corners[i,2], corners[j,2], None]
        fig.add_trace(go.Scatter3d(
            x=xs, y=ys, z=zs,
            mode="lines",
            line=dict(width=2),
            opacity=cube_opacity,
            hoverinfo="skip",
            showlegend=False
        ))

    # High-value isosurfaces
    # Plotly’s Isosurface can take the volume and draw surfaces at given isovalues.
    fig.add_trace(go.Isosurface(
        value=vol.flatten(order="C"),
        x=np.repeat(np.arange(X), Y*Z),
        y=np.tile(np.repeat(np.arange(Y), Z), X),
        z=np.tile(np.arange(Z), X*Y),
        isomin=isovals[0],
        isomax=isovals[-1],
        surface_count=n_isosurfaces,
        # Keep low values invisible; only top tail draws surfaces
        caps=dict(x_show=False, y_show=False, z_show=False),
        opacity=0.3,
        showscale=True
    ))

    fig.update_layout(
        scene=dict(
            xaxis=dict(visible=show_grid),
            yaxis=dict(visible=show_grid),
            zaxis=dict(visible=show_grid),
            aspectmode="data"
        ),
        margin=dict(l=0, r=0, t=0, b=0),   
    )
    if title:
        fig.update_layout(title=str(title))
        # we center it
        fig.update_layout(title_y=0.95)
    if show:
        fig.show()
    if save_dir:
        fig.write_html(save_dir)
    if return_fig:
        return fig
    
# we check the equivalence of both core projection functions
test_tuple = ("spelen", "kind", "bal")
core_proj_slow = create_core_projection(test_tuple, weighted=True)
core_proj_fast = core_projection_fast(test_tuple, weighted=True)
# we use np is all close
print("Core projections equal:", np.allclose(core_proj_slow, core_proj_fast))

In [ ]:
variant_sentences = [("spelen", "man", "rol"), 
                     ("spelen", "man", "spel"), 
                     ("spelen", "kind", "rol"), 
                     ("spelen", "kind", "spel")]

for i, variant in enumerate(variant_sentences):
    core_proj = core_projection_fast(variant)
    core_proj_weighted = core_projection_fast(variant, weighted=True)
    # plot_tensor_plotly(core_proj, title=variant, top_quantile=0.5, n_isosurfaces=100, show_cube=True, show=False, cube_opacity=0.3, save_dir=f"core_projection_{i}.html")
    plot_tensor_plotly(core_proj_weighted, title=variant, top_quantile=0.2, n_isosurfaces=1000, show_cube=True, show=False, cube_opacity=0.3, save_dir=fDATA_DIR/"vis/core_projection_{i}_weighted_all.html")

In [ ]:
core_0 = core_projection_fast(variant_sentences[0], weighted=True)
core_1 = core_projection_fast(variant_sentences[1], weighted=True)
# we calculate the difference, but only over the top quantile
core_diff_0_1 = core_0 - core_1
plot_tensor_plotly(core_diff_0_1, title=f"Difference between {variant_sentences[0]} and {variant_sentences[1]}",
                   top_quantile=0.5, n_isosurfaces=100, show_cube=True, show=False, cube_opacity=0.3, save_dir=DATA_DIR/"vis/core_difference_0_1.html")
core_diff_1_0 = core_1 - core_0
plot_tensor_plotly(core_diff_1_0, title=f"Difference between {variant_sentences[1]} and {variant_sentences[0]}",
                   top_quantile=0.5, n_isosurfaces=100, show_cube=True, show=False, cube_opacity=0.3, save_dir=DATA_DIR/"vis/core_difference_1_0.html")

In [ ]:
core_2 = core_projection_fast(variant_sentences[2], weighted=True)
core_3 = core_projection_fast(variant_sentences[3], weighted=True)
# we calculate the difference, but only over the top quantile
core_diff_2_3 = core_2 - core_3
plot_tensor_plotly(core_diff_2_3, title=f"Difference between {variant_sentences[2]} and {variant_sentences[3]}",
                   top_quantile=0.5, n_isosurfaces=100, show_cube=True, show=False, cube_opacity=0.3, save_dir=DATA_DIR/"vis/core_difference_2_3.html")
core_diff_3_2 = core_3 - core_2
plot_tensor_plotly(core_diff_3_2, title=f"Difference between {variant_sentences[3]} and {variant_sentences[2]}",
                   top_quantile=0.5, n_isosurfaces=100, show_cube=True, show=False, cube_opacity=0.3, save_dir=DATA_DIR/"vis/core_difference_3_2.html")

In [ ]:
from numpy.linalg import norm
cores = [core_0, core_1, core_2, core_3]
for i in range(len(cores)):
    for j in range(i + 1, len(cores)):
        core0 = cores[i]
        core1 = cores[j]
        
        print(f"Comparing core {i} and core {j}:")
        print(f"Sentences: {variant_sentences[i]} vs {variant_sentences[j]}")
        # Flatten tensors
        v0 = core0.flatten()
        v1 = core1.flatten()
        
        # Basic measures
        euclidean = norm(v0 - v1)
        cosine = np.dot(v0, v1) / (norm(v0) * norm(v1))
        manhattan = np.sum(np.abs(v0 - v1))
        relative_change = np.mean(np.abs((v0 - v1) / (v1 + 1e-9)))  # gpt created extension
        
        print("Euclidean:", euclidean)
        print("Cosine similarity:", cosine)
        print("Manhattan:", manhattan)
        print("Relative change:", relative_change)
        # the relative difference is not symmetric, we do it again for the other way around
        relative_change_rev = np.mean(np.abs((v1 - v0) / (v0 + 1e-9)))  # gpt created extension
        print("Reverse relative change:", relative_change_rev)
        print()


## Core projection difference
For a target tuple, calculate which "contextualized" verb would be most similar
--> We calculate all core projections for all verbs with the same subject and object, and see which is most similar to the target core projection

In [ ]:
from tqdm import tqdm
def contextually_similar_verb(target_tuple, distance="cosine"):
    target_core = core_projection_fast(target_tuple, weighted=True)
    subj = target_tuple[1]
    obj = target_tuple[2]
    
    similarities = []
    for verb in tqdm(vocab_v):
        if verb == target_tuple[0]:
            continue
        test_tuple = (verb, subj, obj)
        test_core = core_projection_fast(test_tuple, weighted=True)
        
        if distance == "cosine":
            # Flatten tensors
            v0 = target_core.flatten()
            v1 = test_core.flatten()
            
            # Cosine similarity
            cosine = np.dot(v0, v1)
            similarities.append((verb, cosine))
    
    similarities.sort(key=lambda x: x[1], reverse=True)
    return similarities

def contextually_similar_subj(target_tuple, distance="cosine"):
    target_core = core_projection_fast(target_tuple, weighted=True)
    verb = target_tuple[0]
    obj = target_tuple[2]
    
    similarities = []
    for subj in tqdm(vocab_s):
        if subj == target_tuple[1]:
            continue
        test_tuple = (verb, subj, obj)
        test_core = core_projection_fast(test_tuple, weighted=True)
        
        if distance == "cosine":
            # Flatten tensors
            v0 = target_core.flatten()
            v1 = test_core.flatten()
            
            # Cosine similarity
            cosine = np.dot(v0, v1) # / (norm(v0) * norm(v1))
            similarities.append((subj, cosine))
    
    similarities.sort(key=lambda x: x[1], reverse=True)
    return similarities

def contextually_similar_obj(target_tuple, distance="cosine"):
    target_core = core_projection_fast(target_tuple, weighted=True)
    verb = target_tuple[0]
    subj = target_tuple[1]
    
    similarities = []
    for obj in tqdm(vocab_o):
        if obj == target_tuple[2]:
            continue
        test_tuple = (verb, subj, obj)
        test_core = core_projection_fast(test_tuple, weighted=True)
        
        if distance == "cosine":
            # Flatten tensors
            v0 = target_core.flatten()
            v1 = test_core.flatten()
            
            # Cosine similarity
            cosine = np.dot(v0, v1) # / (norm(v0) * norm(v1))
            similarities.append((obj, cosine))
    
    similarities.sort(key=lambda x: x[1], reverse=True)
    return similarities

# we test with a variant sentence
contextualised_distance = contextually_similar_verb(("spelen", "kind", "spel"))
verb_distance = top_similar_verbs("spelen", top_n=10)
print("Contextually similar verbs (core projection distance):", contextualised_distance[:10])
print("General similar verbs (factor cosine similarity):", verb_distance)

In [ ]:
# we test with a variant sentence
contextualised_distance = contextually_similar_verb(("spelen", "kind", "rol"))
verb_distance = top_similar_verbs("spelen", top_n=10)
print("Contextually similar verbs (core projection distance):", contextualised_distance[:10])
print("General similar verbs (factor cosine similarity):", verb_distance)

In [ ]:
# we test with a variant sentence
contextualised_distance = contextually_similar_verb(("eten", "man", "ontbijt"))
verb_distance = top_similar_verbs("eten", top_n=10)
print("Contextually similar verbs (core projection distance):", contextualised_distance[:10])
print("General similar verbs (factor cosine similarity):", verb_distance)

In [ ]:
# we test with a variant sentence
contextualised_distance = contextually_similar_obj(("eten", "ik", "lunch"))
print("Contextually similar objects (core projection distance):", contextualised_distance[:10])
contextualised_distance = contextually_similar_subj(("eten", "ik", "lunch"))
print("Contextually similar subjects (core projection distance):", contextualised_distance[:10])

In [ ]:
def contextually_similar_verb(target_tuple, distance="cosine"):
    target_core = core_projection_fast(target_tuple, weighted=True)
    subj = target_tuple[1]
    obj = target_tuple[2]
    
    similarities = []
    for verb in tqdm(vocab_v):
        if verb == target_tuple[0]:
            continue
        test_tuple = (verb, subj, obj)
        test_core = core_projection_fast(test_tuple, weighted=True)
        
        if distance == "cosine":
            # Flatten tensors
            v0 = target_core.flatten()
            v1 = test_core.flatten()
            
            # Cosine similarity
            cosine = np.dot(v0, v1) / (norm(v0) * norm(v1))
            similarities.append((verb, cosine))
    
    similarities.sort(key=lambda x: x[1], reverse=True)
    return similarities

# we test with a variant sentence
contextualised_distance = contextually_similar_verb(("spelen", "kind", "spel"))
verb_distance = top_similar_verbs("spelen", top_n=10)
print("Contextually similar verbs (core projection distance):", contextualised_distance[:10])
print("General similar verbs (factor cosine similarity):", verb_distance)

### Mode-wise change

In [ ]:

verb_shift = np.mean(np.abs(core_diff_0_1), axis=(1,2))
subj_shift = np.mean(np.abs(core_diff_0_1), axis=(0,2))
obj_shift = np.mean(np.abs(core_diff_0_1), axis=(0,1))
import matplotlib.pyplot as plt

# Create a figure with three subplots side-by-side
fig, axes = plt.subplots(1, 3, figsize=(18, 6), sharey=True)

# Plot for verb_shift
axes[0].bar(range(len(verb_shift)), verb_shift, color='blue')
axes[0].set_title("Verb Dimension Shift")
axes[0].set_xlabel("Verb Dimensions")
axes[0].set_ylabel("Shift Magnitude")

# Plot for subj_shift
axes[1].bar(range(len(subj_shift)), subj_shift, color='green')
axes[1].set_title("Subject Dimension Shift")
axes[1].set_xlabel("Subject Dimensions")

# Plot for obj_shift
axes[2].bar(range(len(obj_shift)), obj_shift, color='orange')
axes[2].set_title("Object Dimension Shift")
axes[2].set_xlabel("Object Dimensions")

# Adjust layout for better spacing
plt.tight_layout()
plt.show()

In [ ]:
# we print the dimensions for the top highest shifts
def print_top_shifts(shift_array, mode_name, top_n=3):
    top_indices = np.argsort(shift_array)[-top_n:][::-1]
    print(f"Top {top_n} shifting dimensions for {mode_name}:")
    for idx in top_indices:
        print(f"Dimension {idx}: Shift magnitude {shift_array[idx]:.6f}")
        if mode_name == "Verb":
            top_values, top_indices = torch.topk(factor_verb[:, idx], 10)
            top_els = [f"{vocab_v[i]} ({top_values[j].item():.3f})" for j, i in enumerate(top_indices)]
        elif mode_name == "Subject":
            top_values, top_indices = torch.topk(factor_subj[:, idx], 10)
            top_els = [f"{vocab_s[i]} ({top_values[j].item():.3f})" for j, i in enumerate(top_indices)]
        else:  # Object
            top_values, top_indices = torch.topk(factor_obj[:, idx], 10)
            top_els = [f"{vocab_o[i]} ({top_values[j].item():.3f})" for j, i in enumerate(top_indices)]
        print("Top elements in dimension of the shift shift:", top_els)
    print("\n")
print_top_shifts(verb_shift, "Verb")
print_top_shifts(subj_shift, "Subject")
print_top_shifts(obj_shift, "Object")


## Shift vector calculation and visualization
Quick GPT-powered visualisation: not yet checked if the values are actually coherent/meaningful

In [ ]:
# we now create vectors that show the shift between two sentences
import numpy as np

def _centroid_topq(vol, top_q=0.95, ignore_zeros=True):
    """Return value-weighted centroid (x,y,z) over top-quantile voxels."""
    v = vol
    data = v[v > 0] if ignore_zeros else v.ravel()
    if data.size == 0:
        raise ValueError("No positive voxels to compute centroid.")
    thr = np.quantile(data, top_q)

    mask = v >= thr
    if not np.any(mask):
        raise ValueError("Top-quantile mask is empty; lower top_q or check data.")

    z_idx, y_idx, x_idx = np.nonzero(mask)
    w = v[mask]
    wsum = float(w.sum())
    if wsum == 0:
        raise ValueError("Weights sum to zero; check normalization.")

    cx = float((w * x_idx).sum() / wsum)
    cy = float((w * y_idx).sum() / wsum)
    cz = float((w * z_idx).sum() / wsum)
    return np.array([cx, cy, cz], dtype=np.float32)

def shift_vector_centroid(vol_a, vol_b, top_q=0.95, ignore_zeros=True, voxel_size=(1.0,1.0,1.0)):
    """
    Return a 3D shift vector Δ = centroid_b - centroid_a, in *physical* units if voxel_size given.
    Coordinates are (x, y, z).
    """
    cA = _centroid_topq(vol_a, top_q=top_q, ignore_zeros=ignore_zeros)
    cB = _centroid_topq(vol_b, top_q=top_q, ignore_zeros=ignore_zeros)
    # convert voxel -> physical
    vx, vy, vz = voxel_size
    d = (cB - cA) * np.array([vx, vy, vz], dtype=np.float32)
    return d, cA*np.array([vx,vy,vz]), cB*np.array([vx,vy,vz])
import plotly.graph_objects as go

def visualize_shift_vector(vol_shape, start_xyz, vec_xyz, title="Shift vector", cube_opacity=0.2):
    """
    vol_shape: (Z,Y,X)
    start_xyz: (x0,y0,z0) in the same units as vec_xyz (voxel or physical)
    vec_xyz:   (dx,dy,dz)
    """
    Z, Y, X = vol_shape
    fig = go.Figure()

    # cube wireframe
    corners = np.array([[0,0,0],[X,0,0],[X,Y,0],[0,Y,0],[0,0,Z],[X,0,Z],[X,Y,Z],[0,Y,Z]])
    edges = [(0,1),(1,2),(2,3),(3,0),(4,5),(5,6),(6,7),(7,4),(0,4),(1,5),(2,6),(3,7)]
    xs, ys, zs = [], [], []
    for i,j in edges:
        xs += [corners[i,0], corners[j,0], None]
        ys += [corners[i,1], corners[j,1], None]
        zs += [corners[i,2], corners[j,2], None]
    fig.add_trace(go.Scatter3d(x=xs,y=ys,z=zs,mode="lines",opacity=cube_opacity,hoverinfo="skip"))

    # start & end points
    x0,y0,z0 = start_xyz
    dx,dy,dz = vec_xyz
    fig.add_trace(go.Scatter3d(x=[x0, x0+dx], y=[y0, y0+dy], z=[z0, z0+dz],
                               mode="markers+lines", marker=dict(size=4), name="vector path"))

    # arrow (cone) at the head
    fig.add_trace(go.Cone(x=[x0+dx], y=[y0+dy], z=[z0+dz],
                          u=[dx], v=[dy], w=[dz],
                          sizemode="absolute", sizeref=0.2, anchor="tip", showscale=False, name="Δ"))

    fig.update_layout(scene=dict(aspectmode="data"), title=title, margin=dict(l=0,r=0,t=30,b=0))
    # we save as html
    # fig.write_html(DATA_DIR/"vis/shift_vector.html")
    fig.show()



In [ ]:
# Example usage
vol_a = core_projection_fast(variant_sentences[0], weighted=True)
fig_a = plot_tensor_plotly(vol_a, title=variant_sentences[0], top_quantile=0.5, n_isosurfaces=500, show_cube=True, show=False, cube_opacity=0.3, return_fig=True)
vol_b = core_projection_fast(variant_sentences[1], weighted=True)
fig_b = plot_tensor_plotly(vol_b, title=variant_sentences[1], top_quantile=0.5, n_isosurfaces=500, show_cube=True, show=False, cube_opacity=0.3, return_fig=True)

In [ ]:
# Suppose vol_a, vol_b have shape (Z,Y,X)
d_centroid, cA, cB = shift_vector_centroid(vol_a, vol_b, top_q=0.0, voxel_size=(1,1,1))
print("Centroid Δ (x,y,z):", d_centroid)

# Visualize centroid-based shift
visualize_shift_vector(vol_a.shape, start_xyz=cA, vec_xyz=d_centroid, title="Centroid-based shift")


In [ ]:
def add_shift_vector_to_plotly(fig, start_xyz, vec_xyz, color="red", name="Shift Vector", arrow_size=0.3):
    """
    Adds a 3D vector arrow to an existing Plotly figure.
    start_xyz: (x0, y0, z0)
    vec_xyz: (dx, dy, dz)
    """
    x0, y0, z0 = start_xyz
    dx, dy, dz = vec_xyz

    # Line body
    fig.add_trace(go.Scatter3d(
        x=[x0, x0 + dx],
        y=[y0, y0 + dy],
        z=[z0, z0 + dz],
        mode="lines+markers",
        line=dict(color=color, width=6),
        marker=dict(size=4, color=color),
        name=name
    ))

    # Arrow head (cone)
    fig.add_trace(go.Cone(
        x=[x0 + dx],
        y=[y0 + dy],
        z=[z0 + dz],
        u=[dx],
        v=[dy],
        w=[dz],
        sizemode="absolute",
        sizeref=arrow_size,
        anchor="tip",
        colorscale=[[0, color], [1, color]],
        showscale=False,
        name=f"{name} (arrow)"
    ))

add_shift_vector_to_plotly(fig_b, start_xyz=cA, vec_xyz=d_centroid, color="red")

# Show or save
fig_b.show()
fig_b.write_html(DATA_DIR/"vis/volume_b_with_vector.html")
add_shift_vector_to_plotly(fig_a, start_xyz=cA, vec_xyz=d_centroid, color="red")

# Show or save
fig_a.show()
fig_a.write_html(DATA_DIR/"vis/volume_a_with_vector.html")